# XGBoost Training - Occupancy Prediction
Trains an xgboost classifier to predict bus occupancy level (low, medium, high, very_high).

In [25]:
# --- imports ---
import pandas as pd
import numpy as np
import pickle
import time
import warnings
import os 
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from xgboost import XGBClassifier

print("Libraries imported successfully")

Libraries imported successfully


In [26]:
# --- configuration ---

# paths to X and y files generated by the feature engineering notebook
# DATASET_PATH = "./data"
DATASET_PATH = "../data"

# where to save the trained model and results
# MODEL_PATH = "./occupancy"
MODEL_PATH = "/Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy"

USE_LAGS = False
suffix   = "with_lags" if USE_LAGS else "no_lags"

X_PATH = f"{DATASET_PATH}/sunt_2024_4months_{suffix}_X.parquet"
Y_PATH = f"{DATASET_PATH}/sunt_2024_4months_{suffix}_y.pkl"

# train/test split
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# xgboost hyperparameters
N_ESTIMATORS         = 300
MAX_DEPTH            = 8
LEARNING_RATE        = 0.05
SUBSAMPLE            = 0.8
COLSAMPLE            = 0.8
MIN_CHILD_WEIGHT     = 5

# stop training if validation loss does not improve after this many rounds
EARLY_STOPPING_ROUNDS = 20

print(f"X path : {X_PATH}")
print(f"y path : {Y_PATH}")
print(f"model  : {MODEL_PATH}")

X path : ../data/sunt_2024_4months_no_lags_X.parquet
y path : ../data/sunt_2024_4months_no_lags_y.pkl
model  : /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy


In [27]:
# --- load data ---

X = pd.read_parquet(X_PATH)
with open(Y_PATH, 'rb') as f:
    y = pickle.load(f)

print("Dataset loaded:")
print(f"  X shape  : {X.shape}")
print(f"  y shape  : {y.shape}")
print(f"  features : {list(X.columns)}")
print(f"\nTarget distribution:")
dist = y.value_counts(normalize=True).sort_index()
for cls, pct in dist.items():
    count = (y == cls).sum()
    bar = '█' * int(pct * 50)
    print(f"  {cls:12s}: {count:>10,} ({pct*100:5.2f}%) {bar}")

Dataset loaded:
  X shape  : (41835224, 12)
  y shape  : (41835224,)
  features : ['route_short_name', 'direction_id', 'pt_sequence', 'stop_id', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'route_progression', 'trip_stage', 'time_of_day', 'loading_mean_route']

Target distribution:
  high        :  4,556,303 (10.89%) █████
  low         : 25,587,098 (61.16%) ██████████████████████████████
  medium      : 10,505,265 (25.11%) ████████████
  very_high   :  1,186,558 ( 2.84%) █


In [28]:
# --- analyze class imbalance and compute class weights ---
# minority classes get higher weights so xgboost pays more attention to them
# this balances the model without discarding any data

class_counts = y.value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Imbalance ratio : {imbalance_ratio:.1f}:1")
print(f"Majority class  : {class_counts.idxmax()} ({class_counts.max():,})")
print(f"Minority class  : {class_counts.idxmin()} ({class_counts.min():,})")

classes = np.unique(y)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y)
class_weights = dict(zip(classes, class_weights_array))

print(f"\nClass weights:")
for cls, weight in sorted(class_weights.items()):
    print(f"  {cls:12s}: {weight:.4f}")

Imbalance ratio : 21.6:1
Majority class  : low (25,587,098)
Minority class  : very_high (1,186,558)

Class weights:
  high        : 2.2955
  low         : 0.4088
  medium      : 0.9956
  very_high   : 8.8144


In [29]:
# --- encode target and split data ---

# xgboost requires numeric labels (0, 1, 2, 3)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label encoding:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:12s} -> {i}")

# stratified split preserves class distribution in both sets
X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

# keep original string labels for reporting
_, _, y_train_orig, y_test_orig = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"\nTrain set : {len(X_train):,} records")
print(f"Test set  : {len(X_test):,} records")

Label encoding:
  high         -> 0
  low          -> 1
  medium       -> 2
  very_high    -> 3

Train set : 33,468,179 records
Test set  : 8,367,045 records


In [30]:
# --- compute sample weights for training set ---
# one weight per sample, passed to model.fit() to adjust the loss function

sample_weights_train = compute_sample_weight(class_weights, y_train_orig)

print(f"Sample weights computed:")
print(f"  total samples : {len(sample_weights_train):,}")
print(f"  min weight    : {sample_weights_train.min():.4f}")
print(f"  max weight    : {sample_weights_train.max():.4f}")
print(f"  weight ratio  : {sample_weights_train.max() / sample_weights_train.min():.2f}:1")

Sample weights computed:
  total samples : 33,468,179
  min weight    : 0.4088
  max weight    : 8.8144
  weight ratio  : 21.56:1


In [31]:
# --- train xgboost model ---

model = XGBClassifier(
    objective='multi:softprob',
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=0.1,
    reg_lambda=1,
    tree_method='hist',
    eval_metric='mlogloss',
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=1
)

print("XGBoost configuration:")
print(f"  n_estimators (max)    : {N_ESTIMATORS}")
print(f"  max_depth             : {MAX_DEPTH}")
print(f"  learning_rate         : {LEARNING_RATE}")
print(f"  early_stopping_rounds : {EARLY_STOPPING_ROUNDS}")
print(f"  sample_weight         : enabled")
print("\nTraining...")

start_time = time.time()

model.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weights_train,
    # monitor test loss to trigger early stopping
    eval_set=[(X_test, y_test_enc)],
    verbose=50
)

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.2f}s ({elapsed/60:.1f} min)")
print(f"Best iteration: {model.best_iteration} (out of {N_ESTIMATORS} max)")

XGBoost configuration:
  n_estimators (max)    : 300
  max_depth             : 8
  learning_rate         : 0.05
  early_stopping_rounds : 20
  sample_weight         : enabled

Training...
[0]	validation_0-mlogloss:1.36915
[50]	validation_0-mlogloss:1.07090
[100]	validation_0-mlogloss:0.99967
[150]	validation_0-mlogloss:0.96575
[200]	validation_0-mlogloss:0.94397
[250]	validation_0-mlogloss:0.92990
[299]	validation_0-mlogloss:0.91905

Training completed in 1152.98s (19.2 min)
Best iteration: 299 (out of 300 max)


In [ ]:
# --- evaluate model ---
# primary metric: f1 macro (treats all classes equally regardless of size)
# secondary metric: balanced accuracy (average recall per class)

y_pred_train = label_encoder.inverse_transform(model.predict(X_train))
y_pred_test  = label_encoder.inverse_transform(model.predict(X_test))

metrics = {
    'accuracy'         : (accuracy_score(y_train_orig, y_pred_train),          accuracy_score(y_test_orig, y_pred_test)),
    'balanced accuracy': (balanced_accuracy_score(y_train_orig, y_pred_train),  balanced_accuracy_score(y_test_orig, y_pred_test)),
    'f1 macro'         : (f1_score(y_train_orig, y_pred_train, average='macro'), f1_score(y_test_orig, y_pred_test, average='macro')),
}

print(f"{'Metric':<22} {'Train':>10} {'Test':>10} {'Gap':>10}")
print('-' * 55)
for name, (train_val, test_val) in metrics.items():
    print(f"{name:<22} {train_val:>10.4f} {test_val:>10.4f} {train_val - test_val:>10.4f}")

gap = metrics['balanced accuracy'][0] - metrics['balanced accuracy'][1]
if gap > 0.1:
    print(f"\nWarning: high gap ({gap:.2%}) — possible overfitting")
elif gap > 0.05:
    print(f"\nModerate gap ({gap:.2%}) — acceptable")
else:
    print(f"\nLow gap ({gap:.2%}) — model generalizes well")

In [ ]:
# --- classification report ---
print("Classification Report (test set):")
print(classification_report(y_test_orig, y_pred_test))

Classification Report (test set):
              precision    recall  f1-score   support

        high       0.87      0.89      0.88    882967
         low       0.98      0.96      0.97   4711742
      medium       0.88      0.90      0.89   2028551
   very_high       0.84      0.94      0.89    229821

    accuracy                           0.93   7853081
   macro avg       0.89      0.92      0.90   7853081
weighted avg       0.93      0.93      0.93   7853081



In [ ]:
# --- confusion matrix ---

labels = sorted(y.unique())
cm = confusion_matrix(y_test_orig, y_pred_test, labels=labels)

print("Confusion Matrix (test set):")
print(f"\n{'':>12}", end='')
for label in labels:
    print(f"{label:>12}", end='')
print("  <- predicted")
print('-' * (12 + 12 * len(labels)))
for i, label in enumerate(labels):
    print(f"{label:>12}", end='')
    for j in range(len(labels)):
        print(f"{cm[i,j]:>12,}", end='')
    print("  | actual")

print("\nPer-class accuracy:")
for i, label in enumerate(labels):
    total   = cm[i, :].sum()
    correct = cm[i, i]
    acc     = correct / total if total > 0 else 0
    status  = 'OK' if acc > 0.6 else 'LOW' if acc > 0.4 else 'POOR'
    print(f"  {status:4s} {label:12s}: {acc:.2%} ({correct:,}/{total:,})")

Confusion Matrix (test set):

                    high         low      medium   very_high  <- predicted
------------------------------------------------------------
        high     786,062       1,613      55,745      39,547  | actual
         low       5,803   4,508,562     196,601         776  | actual
      medium     100,614     108,170   1,818,273       1,494  | actual
   very_high      13,227          57         465     216,072  | actual

Per-class accuracy:
  OK   high        : 89.03% (786,062/882,967)
  OK   low         : 95.69% (4,508,562/4,711,742)
  OK   medium      : 89.63% (1,818,273/2,028,551)
  OK   very_high   : 94.02% (216,072/229,821)


In [ ]:
# --- feature importance ---

importances   = model.feature_importances_
feature_names = X.columns.tolist()
indices       = np.argsort(importances)[::-1]

print("Top features by importance:")
print('-' * 55)
for i in range(min(15, len(feature_names))):
    idx = indices[i]
    bar = '█' * int(importances[idx] * 50)
    print(f"{i+1:2d}. {feature_names[idx]:<25} {importances[idx]:.4f}  {bar}")

cumsum = 0
for i, idx in enumerate(indices):
    cumsum += importances[idx]
    if cumsum >= 0.8:
        print(f"\nTop {i+1} features explain 80% of total importance")
        break

Top features by importance:
-------------------------------------------------------
 1. loading_lag_1             0.7516  █████████████████████████████████████
 2. loading_lag_2             0.1935  █████████
 3. route_progression         0.0095  
 4. trip_stage                0.0094  
 5. hour                      0.0065  
 6. direction_id              0.0055  
 7. pt_sequence               0.0044  
 8. stop_id                   0.0044  
 9. loading_mean_route        0.0031  
10. route_short_name          0.0031  
11. time_of_day               0.0026  
12. is_weekend                0.0022  
13. day_of_week               0.0022  
14. is_rush_hour              0.0019  

Top 2 features explain 80% of total importance


In [ ]:
import os 

In [ ]:
# --- compare with baseline and random forest ---
# baseline: always predict the majority class (simplest possible model)
# rf results are loaded from file if available (saved by the rf training notebook)
# otherwise falls back to hardcoded values from the original run

majority_class = y_train_orig.value_counts().idxmax()
baseline_acc   = (y_test_orig == majority_class).mean()

RF_RESULTS_PATH = f"{MODEL_PATH}/rf_results_{suffix}.pkl"
if os.path.exists(RF_RESULTS_PATH):
    with open(RF_RESULTS_PATH, 'rb') as f:
        rf_results  = pickle.load(f)
    rf_acc      = rf_results['accuracy']
    rf_bal_acc  = rf_results['balanced_accuracy']
    rf_f1_macro = rf_results['f1_macro']
    print("RF results loaded from file")
else:
    rf_acc      = 0.5920
    rf_bal_acc  = 0.5677
    rf_f1_macro = 0.4753
    print("RF results using hardcoded values (run TrainingRandomForest first to update)")

xgb_acc      = metrics['accuracy'][1]
xgb_bal_acc  = metrics['balanced accuracy'][1]
xgb_f1_macro = metrics['f1 macro'][1]

print(f"\n{'Model':<20} {'Accuracy':>10} {'Bal. Acc':>10} {'F1 Macro':>10}")
print('-' * 55)
print(f"{'Baseline':<20} {baseline_acc:>10.4f} {'—':>10} {'—':>10}")
print(f"{'Random Forest':<20} {rf_acc:>10.4f} {rf_bal_acc:>10.4f} {rf_f1_macro:>10.4f}")
print(f"{'XGBoost':<20} {xgb_acc:>10.4f} {xgb_bal_acc:>10.4f} {xgb_f1_macro:>10.4f}")
print(f"\nXGBoost vs Random Forest:")
print(f"  balanced accuracy : {(xgb_bal_acc - rf_bal_acc)*100:+.2f}%")
print(f"  f1 macro          : {(xgb_f1_macro - rf_f1_macro)*100:+.2f}%")

RF results loaded from file

Model                  Accuracy   Bal. Acc   F1 Macro
-------------------------------------------------------
Baseline                 0.6000          —          —
Random Forest            0.9344     0.9190     0.9067
XGBoost                  0.9333     0.9209     0.9046

XGBoost vs Random Forest:
  balanced accuracy : +0.19%
  f1 macro          : -0.20%


In [ ]:
# --- save model and results to drive ---
model_file = f"{MODEL_PATH}/xgboost_occupancy_{suffix}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved        -> {model_file}")

encoder_file = f"{MODEL_PATH}/xgboost_label_encoder_{suffix}.pkl"
with open(encoder_file, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder      -> {encoder_file}")

features_file = f"{MODEL_PATH}/xgboost_feature_names_{suffix}.pkl"
with open(features_file, 'wb') as f:
    pickle.dump(feature_names, f)
print(f"Feature names      -> {features_file}")

xgb_results = {
    'accuracy'         : xgb_acc,
    'balanced_accuracy': xgb_bal_acc,
    'f1_macro'         : xgb_f1_macro,
}
xgb_results_file = f"{MODEL_PATH}/xgb_results_{suffix}.pkl"
with open(xgb_results_file, 'wb') as f:
    pickle.dump(xgb_results, f)
print(f"XGBoost results    -> {xgb_results_file}")

Model saved        -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/xgboost_occupancy_with_lags.pkl
Label encoder      -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/xgboost_label_encoder_with_lags.pkl
Feature names      -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/xgboost_feature_names_with_lags.pkl
XGBoost results    -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/xgb_results_with_lags.pkl


In [ ]:
# --- push changes to github ---
!git add 04_trainingXGBoost.ipynb
!git commit -m "xgboost training with 4 months and USE_LAGS = False"
!git push origin floppy

[floppy 562f94d] xgboost training with 4 months
 Committer: Florencia Gonzalez <florenciagonzalez@Florencias-MacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 1 file changed, 108 insertions(+), 86 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 10 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.09 KiB | 2.09 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/victoriaeleonor/bus-pred